# Dengue forecasting: evidence-based model selection and tuning

This notebook lets the data choose the model. Weekly case counts per ward, forecast 2, 3 and 4 weeks ahead.

## How to run it
1. **Runtime > Run all** with `QUICK_RUN = True` (Segment 2). This is a short smoke test that catches errors. Its results are not valid.
2. If it finishes, set `QUICK_RUN = False` and **Run all** again. This is the real run and can take a few hours on a free CPU.
3. Send back `model_selection_report.json` (Segment 14), or paste the printed report.

If Colab disconnects, reconnect and **Run all**. Finished work is saved to Google Drive and is skipped.

## Rules fixed before seeing any result
- **2024-25 is deleted at load.** This notebook cannot see the final test years.
- **Model comparison and tuning use 2016-2022 only**, with 4 rolling yearly folds (2019, 2020, 2021, 2022) and a 4-week purge.
- **Primary metric:** mean Poisson deviance, averaged over horizons and folds. It is the proper score for an expected count.
- **Must beat baselines:** lower MAE than the best simple baseline at every horizon, and in at least 75% of horizon-fold cells.
- **Tie rule:** models within one standard error of the leader are treated as tied, and the simplest one wins.
- **Tuning:** the top 3 models get the same Optuna budget. A tuned model must win under the same rules.
- **2023 is a one-time confirmation** of the winner. It is never used to pick between models.

## Segment 1: Install packages
Colab restarts once after this cell. With Run all, choose **Run all** again after the restart.

In [ ]:
import importlib.metadata as metadata
import os
import subprocess
import sys
from pathlib import Path

CORE = {'numpy': '2.5.5', 'pandas': '3.1.1', 'scikit-learn': '1.9.1', 'lightgbm': '4.7.0', 'joblib': '1.6.0'}
OPTIONAL = ['matplotlib', 'optuna', 'statsmodels', 'xgboost', 'catboost']
MARKER = Path('/content/.dengue_selection_installed')

if 'google.colab' in sys.modules and not MARKER.exists():
    pins = [f'{name}=={version}' for name, version in CORE.items()]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pins])
    constraints = Path('/content/dengue_constraints.txt')
    constraints.write_text(chr(10).join(pins) + chr(10), encoding='utf-8')
    for package in OPTIONAL:
        command = [sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraints), package]
        done = subprocess.run(command, capture_output=True, text=True)
        print(package, 'installed' if done.returncode == 0 else 'NOT AVAILABLE with the pinned versions')
    MARKER.write_text('done', encoding='utf-8')
    print('Restarting the runtime. Run all again after it reconnects.')
    os.kill(os.getpid(), 9)

print('Python', sys.version.split()[0])
for name in [*CORE, *OPTIONAL, 'torch']:
    try:
        print(f'{name}: {metadata.version(name)}')
    except metadata.PackageNotFoundError:
        print(f'{name}: not installed')

## Segment 2: Settings
Keep `QUICK_RUN = True` for the first run. Set it to `False` for the real run.

`USE_DRIVE = True` asks for Google Drive access so progress survives a disconnect. Files go in `MyDrive/dengue_model_selection`.

In [ ]:
QUICK_RUN = True
USE_DRIVE = True

import importlib
import json
import os
import re
import shutil
import sys
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import PoissonRegressor, TweedieRegressor
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance, mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler

warnings.filterwarnings('ignore')


def optional_import(name):
    try:
        return importlib.import_module(name)
    except Exception as exc:
        print(f'{name} unavailable: {type(exc).__name__}')
        return None


xgb = optional_import('xgboost')
cb = optional_import('catboost')
sm = optional_import('statsmodels.api')
optuna = optional_import('optuna')
torch = optional_import('torch')

SEED = 42
CPUS = os.cpu_count() or 2
PURGE = pd.Timedelta(weeks=4)
LAST_ALLOWED = pd.Timestamp('2023-12-31')
DEV_END = pd.Timestamp('2022-12-31')
DEV_FLOOR = 1e-3
TIER_QUANTILES = (0.60, 0.85, 0.95)

HORIZONS = (2,) if QUICK_RUN else (2, 3, 4)
VAL_YEARS = (2022,) if QUICK_RUN else (2019, 2020, 2021, 2022)
MAX_TREES = 60 if QUICK_RUN else 3000
HGB_MAX_ITER = 60 if QUICK_RUN else 800
ES_ROUNDS = 10 if QUICK_RUN else 100
FOREST_TREES = 20 if QUICK_RUN else 300
GLM_ITER = 100 if QUICK_RUN else 1000
MAX_EPOCHS = 2 if QUICK_RUN else 60
PATIENCE = 1 if QUICK_RUN else 8
N_TRIALS = 2 if QUICK_RUN else 40
TUNE_MINUTES = 3 if QUICK_RUN else 45

IN_COLAB = 'google.colab' in sys.modules
BASE = Path('/content') if IN_COLAB else Path.cwd()
if IN_COLAB and USE_DRIVE:
    from google.colab import drive

    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive')
PROJECT = BASE / 'dengue_model_selection'
RUN_DIR = PROJECT / ('quick' if QUICK_RUN else 'full')
CV_DIR = RUN_DIR / 'cv'
TUNE_DIR = RUN_DIR / 'tuning'
for folder in (CV_DIR, TUNE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
print('MODE:', 'QUICK SMOKE TEST (results are not valid)' if QUICK_RUN else 'FULL RUN')
print('Saving to:', RUN_DIR, '| CPUs:', CPUS)

## Segment 3: Load the data and remove the final test years
Upload `ml/data/dengue.csv` when asked. It is only requested once when Drive is on.

In [ ]:
DATA_FILE = PROJECT / 'dengue.csv'
if not DATA_FILE.exists():
    if Path('/content/dengue.csv').exists():
        shutil.copy('/content/dengue.csv', DATA_FILE)
    elif IN_COLAB:
        from google.colab import files

        uploaded = files.upload()
        DATA_FILE.write_bytes(uploaded[next(iter(uploaded))])
    else:
        raise FileNotFoundError(f'Put dengue.csv at {DATA_FILE}')

DATE, CITY, WARD, TARGET = 'date', 'city', 'ward_id', 'dengue_cases'
STATIC = ['population_density_sqkm', 'water_logging_risk_index', 'sanitation_score']
WEATHER = ['avg_temp_c', 'max_temp_c', 'min_temp_c', 'rainfall_mm', 'relative_humidity_pct']

raw = pd.read_csv(DATA_FILE, encoding='utf-8')
raw.columns = [str(column).strip().lower() for column in raw.columns]
required = [DATE, CITY, WARD, *STATIC, *WEATHER, TARGET]
missing = [column for column in required if column not in raw.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')
raw = raw[required].copy()
raw[DATE] = pd.to_datetime(raw[DATE], errors='coerce')
for column in [*STATIC, *WEATHER, TARGET]:
    raw[column] = pd.to_numeric(raw[column], errors='coerce')
if raw.isna().any().any():
    raise ValueError('The data has missing or invalid values.')
if raw.duplicated([WARD, DATE]).any():
    raise ValueError('The data has duplicate ward-weeks.')
raw[WARD] = raw[WARD].astype(str).str.strip()
raw[CITY] = raw[CITY].astype(str).str.strip()
raw = raw.sort_values([WARD, DATE], kind='stable').reset_index(drop=True)

hidden_rows = int((raw[DATE] > LAST_ALLOWED).sum())
raw = raw[raw[DATE] <= LAST_ALLOWED].reset_index(drop=True)
assert raw[DATE].max() <= LAST_ALLOWED
print(f'Kept {len(raw):,} rows through {raw[DATE].max():%Y-%m-%d}. Removed {hidden_rows:,} final-test rows from memory.')
print(f'Wards: {raw[WARD].nunique()} | weeks: {raw[DATE].nunique()}')

## Segment 4: Build features
The same 57 features as `ml/src/features.py`: per-ward weather lags 0-8, rolling rain and means, ward traits and the target week. Case counts are stored only for the target and baselines.

In [ ]:
LAGS = range(9)
RAIN = 'rainfall_mm'
MEAN_COLUMNS = ('avg_temp_c', 'relative_humidity_pct')
WEEKS_PER_YEAR = 52.1775
LAG_FEATURES = [f'{column}_lag{lag}' for column in WEATHER for lag in LAGS]
RAIN_FEATURES = [f'{RAIN}_sum_{window}w' for window in (2, 4, 8)]
MEAN_FEATURES = [f'{column}_mean_{window}w' for column in MEAN_COLUMNS for window in (4, 8)]
FEATURES = [*LAG_FEATURES, *RAIN_FEATURES, *MEAN_FEATURES, *STATIC, 'target_week_sin', 'target_week_cos']
SEQ_COLUMNS = [f'{column}_lag{lag}' for lag in range(8, -1, -1) for column in WEATHER]
OTHER_COLUMNS = [column for column in FEATURES if column not in SEQ_COLUMNS]


def build_frame(horizon, source=None):
    data = (raw if source is None else source).sort_values([WARD, DATE], kind='stable').reset_index(drop=True)
    steps = data.groupby(WARD, sort=False)[DATE].diff().dropna()
    if (steps != pd.Timedelta(weeks=1)).any():
        raise ValueError('Dates must be consecutive weeks within each ward.')
    by_ward = data.groupby(WARD, sort=False)
    values = {}
    for column in WEATHER:
        for lag in LAGS:
            values[f'{column}_lag{lag}'] = by_ward[column].shift(lag)
    for window in (2, 4, 8):
        values[f'{RAIN}_sum_{window}w'] = by_ward[RAIN].transform(lambda s: s.rolling(window, min_periods=window).sum())
    for column in MEAN_COLUMNS:
        for window in (4, 8):
            values[f'{column}_mean_{window}w'] = by_ward[column].transform(lambda s: s.rolling(window, min_periods=window).mean())
    for column in STATIC:
        values[column] = data[column]
    target_date = data[DATE] + pd.Timedelta(weeks=horizon)
    angle = 2 * np.pi * target_date.dt.isocalendar().week.astype(int).to_numpy() / WEEKS_PER_YEAR
    values['target_week_sin'] = pd.Series(np.sin(angle), index=data.index)
    values['target_week_cos'] = pd.Series(np.cos(angle), index=data.index)
    out = pd.concat([data[[DATE, CITY, WARD]], pd.DataFrame(values)[FEATURES]], axis=1)
    out['target_date'] = target_date
    out['target'] = by_ward[TARGET].shift(-horizon)
    out['cases_t'] = data[TARGET]
    out['last_year'] = by_ward[TARGET].shift(52 - horizon)
    return out.dropna(subset=[*FEATURES, 'target']).reset_index(drop=True)


FRAMES = {horizon: build_frame(horizon) for horizon in HORIZONS}

cut = pd.Timestamp('2021-06-28')
keys = [WARD, DATE]
full_check = build_frame(2).set_index(keys)
part_check = build_frame(2, raw[raw[DATE] <= cut]).set_index(keys)
same = np.allclose(full_check.loc[part_check.index, FEATURES].to_numpy(float), part_check[FEATURES].to_numpy(float))
assert same, 'Future rows changed past features'
assert not any('case' in column or column == 'target' for column in FEATURES)
for horizon, frame in FRAMES.items():
    assert (frame['target_date'] - frame[DATE]).eq(pd.Timedelta(weeks=horizon)).all()
    assert frame['target_date'].max() <= LAST_ALLOWED
print(f'{len(FEATURES)} features. Leakage checks passed.')

## Segment 5: What does the data require from a model?
These diagnostics use 2016-2022 only. Each finding points to the model families worth testing.

In [ ]:
dev_raw = raw[raw[DATE] <= DEV_END].copy()
dev_raw['week'] = dev_raw[DATE].dt.isocalendar().week.astype(int)
dev_raw['seasonal_mu'] = dev_raw.groupby([WARD, 'week'])[TARGET].transform('mean')
y = dev_raw[TARGET].to_numpy(float)
mu = dev_raw['seasonal_mu'].to_numpy(float)
usable = mu >= 0.05

dev_raw['residual'] = dev_raw[TARGET] - dev_raw['seasonal_mu']
residual_acf = {}
for lag in range(1, 5):
    residual_acf[lag] = float(dev_raw['residual'].corr(dev_raw.groupby(WARD)['residual'].shift(lag)))

diag_h = 3 if 3 in FRAMES else HORIZONS[0]
diag = FRAMES[diag_h]
diag = diag[diag['target_date'] <= DEV_END]
rain3 = diag['rainfall_mm_lag3']
temp3 = diag['avg_temp_c_lag3']
corr = diag[FEATURES].corr().abs().to_numpy()
upper = corr[np.triu_indices_from(corr, k=1)]
high_rain = rain3 >= rain3.median()
high_temp = temp3 >= temp3.median()
interaction = {
    'dry_cool': float(diag.loc[~high_rain & ~high_temp, 'target'].mean()),
    'dry_warm': float(diag.loc[~high_rain & high_temp, 'target'].mean()),
    'wet_cool': float(diag.loc[high_rain & ~high_temp, 'target'].mean()),
    'wet_warm': float(diag.loc[high_rain & high_temp, 'target'].mean()),
}
ward_means = dev_raw.groupby(WARD)[TARGET].mean()
year_means = dev_raw.groupby(dev_raw[DATE].dt.year)[TARGET].mean()

DIAGNOSTICS = {
    'target_mean': float(y.mean()),
    'target_variance': float(y.var()),
    'variance_to_mean': float(y.var() / y.mean()),
    'seasonal_pearson_dispersion': float(np.mean((y[usable] - mu[usable]) ** 2 / mu[usable])),
    'zero_share_observed': float((y == 0).mean()),
    'zero_share_expected_poisson': float(np.exp(-mu).mean()),
    'residual_autocorrelation': residual_acf,
    'rain_lag3_pearson': float(rain3.corr(diag['target'])),
    'rain_lag3_spearman': float(rain3.corr(diag['target'], method='spearman')),
    'mean_cases_by_rain_and_temp': interaction,
    'feature_pairs_abs_corr_over_0_9': float((upper > 0.9).mean()),
    'ward_mean_min': float(ward_means.min()),
    'ward_mean_max': float(ward_means.max()),
    'year_mean_min': float(year_means.min()),
    'year_mean_max': float(year_means.max()),
}

evidence = pd.DataFrame([
    ['Count target, no negatives', f"mean {DIAGNOSTICS['target_mean']:.2f}", 'Poisson-type losses'],
    ['Overdispersion after season and ward', f"Pearson dispersion {DIAGNOSTICS['seasonal_pearson_dispersion']:.2f} (1 = Poisson)", 'Negative Binomial, Tweedie'],
    ['Excess zeros', f"{DIAGNOSTICS['zero_share_observed']:.1%} observed vs {DIAGNOSTICS['zero_share_expected_poisson']:.1%} expected", 'Hurdle model'],
    ['Time dependence left after season', f'lag-1 residual correlation {residual_acf[1]:.2f}', 'GRU on the weather sequence'],
    ['Non-linear weather effect', f"rain Pearson {DIAGNOSTICS['rain_lag3_pearson']:.2f}, Spearman {DIAGNOSTICS['rain_lag3_spearman']:.2f}", 'Splines, trees, boosting, neural nets'],
    ['Weather interactions', f"wet+warm {interaction['wet_warm']:.2f} vs dry+cool {interaction['dry_cool']:.2f} cases", 'Trees, boosting, neural nets'],
    ['Correlated inputs', f"{DIAGNOSTICS['feature_pairs_abs_corr_over_0_9']:.1%} of feature pairs over 0.9", 'Regularization, tree models'],
    ['Ward differences', f"ward means {DIAGNOSTICS['ward_mean_min']:.2f} to {DIAGNOSTICS['ward_mean_max']:.2f}", 'Ward traits as inputs'],
    ['Year-to-year shift', f"year means {DIAGNOSTICS['year_mean_min']:.2f} to {DIAGNOSTICS['year_mean_max']:.2f}", 'Rolling yearly validation'],
], columns=['Data property', 'Evidence', 'What it points to'])
display(evidence)

deciles = pd.qcut(rain3, 10, duplicates='drop')
curve = diag.groupby(deciles, observed=True)['target'].mean()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].bar(range(len(curve)), curve.to_numpy(), color='#2a78d6')
axes[0].set_title('Mean future cases by rain 3 weeks earlier (low to high group)')
axes[0].set_xlabel('Rain group')
axes[0].set_ylabel('Mean cases')
axes[1].bar(list(residual_acf), list(residual_acf.values()), color='#1f9d8a')
axes[1].set_title('Correlation left after removing ward-season average')
axes[1].set_xlabel('Weeks apart')
plt.tight_layout()
plt.show()

## Segment 6: Candidate models

| Family | Models tested | Why it is valid here |
|---|---|---|
| Count GLM | Poisson, Tweedie, Negative Binomial | Simple, built for counts and overdispersion |
| GLM + splines | Poisson with splines | Non-linear but still interpretable |
| Bagged trees | Random Forest, Extra Trees (Poisson) | Robust to noise and correlated inputs |
| Boosting | HistGradientBoosting, LightGBM (Poisson, Tweedie), XGBoost, CatBoost | Strong on tabular interactions |
| Zero-aware | LightGBM hurdle | Models zero weeks and case size separately |
| Neural | Poisson MLP, Poisson GRU | Flexible; GRU reads weather as a sequence |

**Not tested, with reasons:**
- **SARIMA, ETS, Prophet:** they need each ward's recent case counts at forecast time, which our weather-driven product does not receive, and they need one fitted model per ward.
- **Ordinary linear regression:** can predict negative cases.
- **KNN and SVR:** no count likelihood and poor scaling with 57 correlated inputs.
- **Full zero-inflated GLM:** unstable with this many correlated inputs. The hurdle model covers the same idea.
- **Transformers:** too little data per ward to justify them.

In [ ]:
def combined(Xf, yf, Xe, ye):
    return pd.concat([Xf, Xe], ignore_index=True), np.concatenate([yf, ye])


def fit_poisson_glm(Xf, yf, Xe, ye, Xv, params):
    X, y = combined(Xf, yf, Xe, ye)
    model = make_pipeline(StandardScaler(), PoissonRegressor(alpha=params.get('alpha', 1e-3), max_iter=GLM_ITER))
    return model.fit(X, y).predict(Xv)


def fit_tweedie_glm(Xf, yf, Xe, ye, Xv, params):
    X, y = combined(Xf, yf, Xe, ye)
    regressor = TweedieRegressor(power=params.get('power', 1.5), link='log', alpha=params.get('alpha', 1e-3), max_iter=GLM_ITER)
    return make_pipeline(StandardScaler(), regressor).fit(X, y).predict(Xv)


def fit_spline_glm(Xf, yf, Xe, ye, Xv, params):
    X, y = combined(Xf, yf, Xe, ye)
    spline = SplineTransformer(n_knots=params.get('n_knots', 5), degree=3, include_bias=False, extrapolation='constant')
    regressor = PoissonRegressor(alpha=params.get('alpha', 1e-2), max_iter=GLM_ITER)
    return make_pipeline(StandardScaler(), spline, regressor).fit(X, y).predict(Xv)


def fit_negative_binomial(Xf, yf, Xe, ye, Xv, params):
    if sm is None:
        raise RuntimeError('statsmodels is not available')
    X, y = combined(Xf, yf, Xe, ye)
    scaler = StandardScaler().fit(X)
    A = sm.add_constant(scaler.transform(X), has_constant='add')
    B = sm.add_constant(scaler.transform(Xv), has_constant='add')
    penalty = np.full(A.shape[1], params.get('alpha', 1e-3))
    penalty[0] = 0.0
    start = sm.GLM(y, A, family=sm.families.Poisson()).fit_regularized(alpha=penalty, L1_wt=0.0)
    fitted = np.exp(np.clip(A @ np.asarray(start.params), -20, 8))
    dispersion = max(float(((y - fitted) ** 2 - fitted).sum() / (fitted ** 2).sum()), 1e-3)
    family = sm.families.NegativeBinomial(alpha=dispersion)
    final = sm.GLM(y, A, family=family).fit_regularized(alpha=penalty, L1_wt=0.0, start_params=np.asarray(start.params))
    return np.exp(np.clip(B @ np.asarray(final.params), -20, 8))


def fit_random_forest(Xf, yf, Xe, ye, Xv, params):
    X, y = combined(Xf, yf, Xe, ye)
    args = {'min_samples_leaf': 10, 'max_features': 0.5, 'max_samples': 0.7, **params}
    model = RandomForestRegressor(n_estimators=FOREST_TREES, criterion='poisson', bootstrap=True, n_jobs=CPUS, random_state=SEED, **args)
    return model.fit(X, y).predict(Xv)


def fit_extra_trees(Xf, yf, Xe, ye, Xv, params):
    X, y = combined(Xf, yf, Xe, ye)
    args = {'min_samples_leaf': 10, 'max_features': 0.5, 'max_samples': 0.7, **params}
    model = ExtraTreesRegressor(n_estimators=FOREST_TREES, criterion='poisson', bootstrap=True, n_jobs=CPUS, random_state=SEED, **args)
    return model.fit(X, y).predict(Xv)


def fit_hist_gb(Xf, yf, Xe, ye, Xv, params):
    X, y = combined(Xf, yf, Xe, ye)
    args = {'learning_rate': 0.05, 'max_leaf_nodes': 31, 'min_samples_leaf': 50, 'l2_regularization': 1.0, **params}
    first = HistGradientBoostingRegressor(loss='poisson', max_iter=HGB_MAX_ITER, early_stopping=False, random_state=SEED, **args).fit(Xf, yf)
    curve = [mean_poisson_deviance(ye, np.clip(p, DEV_FLOOR, None)) for p in first.staged_predict(Xe)]
    best = int(np.argmin(curve)) + 1
    final = HistGradientBoostingRegressor(loss='poisson', max_iter=best, early_stopping=False, random_state=SEED, **args)
    return final.fit(X, y).predict(Xv)


LGB_BASE = {'random_state': SEED, 'n_jobs': CPUS, 'verbosity': -1, 'subsample_freq': 1, 'deterministic': True, 'force_row_wise': True}
LGB_DEFAULT = {'learning_rate': 0.03, 'num_leaves': 31, 'min_child_samples': 50, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0}


def lgb_stop_refit(model_class, objective, metric, Xf, yf, Xe, ye, params):
    args = {**LGB_BASE, **objective, **LGB_DEFAULT, **params}
    first = model_class(n_estimators=MAX_TREES, **args)
    first.fit(Xf, yf, eval_X=Xe, eval_y=ye, eval_metric=metric, callbacks=[lgb.early_stopping(ES_ROUNDS, verbose=False)])
    best = max(int(first.best_iteration_ or 1), 1)
    X, y = combined(Xf, yf, Xe, ye)
    return model_class(n_estimators=best, **args).fit(X, y)


def fit_lgb_poisson(Xf, yf, Xe, ye, Xv, params):
    return lgb_stop_refit(lgb.LGBMRegressor, {'objective': 'poisson'}, 'poisson', Xf, yf, Xe, ye, params).predict(Xv)


def fit_lgb_tweedie(Xf, yf, Xe, ye, Xv, params):
    args = {'tweedie_variance_power': 1.5, **params}
    return lgb_stop_refit(lgb.LGBMRegressor, {'objective': 'tweedie'}, 'tweedie', Xf, yf, Xe, ye, args).predict(Xv)


def fit_lgb_hurdle(Xf, yf, Xe, ye, Xv, params):
    classifier = lgb_stop_refit(lgb.LGBMClassifier, {'objective': 'binary'}, 'binary_logloss', Xf, (yf > 0).astype(int), Xe, (ye > 0).astype(int), params)
    fit_positive, stop_positive = yf > 0, ye > 0
    regressor = lgb_stop_refit(lgb.LGBMRegressor, {'objective': 'poisson'}, 'poisson', Xf[fit_positive], yf[fit_positive], Xe[stop_positive], ye[stop_positive], params)
    return classifier.predict_proba(Xv)[:, 1] * regressor.predict(Xv)


def fit_xgboost(Xf, yf, Xe, ye, Xv, params):
    if xgb is None:
        raise RuntimeError('xgboost is not available')
    args = {'learning_rate': 0.03, 'max_depth': 6, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0, **params}
    common = {'objective': 'count:poisson', 'tree_method': 'hist', 'random_state': SEED, 'n_jobs': CPUS}
    first = xgb.XGBRegressor(n_estimators=MAX_TREES, eval_metric='poisson-nloglik', early_stopping_rounds=ES_ROUNDS, **common, **args)
    first.fit(Xf, yf, eval_set=[(Xe, ye)], verbose=False)
    X, y = combined(Xf, yf, Xe, ye)
    return xgb.XGBRegressor(n_estimators=int(first.best_iteration) + 1, **common, **args).fit(X, y).predict(Xv)


def fit_catboost(Xf, yf, Xe, ye, Xv, params):
    if cb is None:
        raise RuntimeError('catboost is not available')
    args = {'learning_rate': 0.03, 'depth': 6, 'l2_leaf_reg': 3.0, 'subsample': 0.8, 'rsm': 0.8, **params}
    common = {'loss_function': 'Poisson', 'bootstrap_type': 'Bernoulli', 'random_seed': SEED, 'thread_count': CPUS, 'verbose': False, 'allow_writing_files': False}
    first = cb.CatBoostRegressor(iterations=MAX_TREES, od_type='Iter', od_wait=ES_ROUNDS, **common, **args)
    first.fit(Xf, yf, eval_set=(Xe, ye), use_best_model=True)
    X, y = combined(Xf, yf, Xe, ye)
    return cb.CatBoostRegressor(iterations=max(int(first.tree_count_), 1), **common, **args).fit(X, y).predict(Xv)


if torch is not None:
    class PoissonMLP(torch.nn.Module):
        def __init__(self, inputs, hidden, layers, dropout):
            super().__init__()
            parts, size = [], inputs
            for _ in range(layers):
                parts += [torch.nn.Linear(size, hidden), torch.nn.ReLU(), torch.nn.Dropout(dropout)]
                size = hidden
            self.net = torch.nn.Sequential(*parts, torch.nn.Linear(size, 1))

        def forward(self, x):
            return self.net(x).squeeze(-1)


    class PoissonGRU(torch.nn.Module):
        def __init__(self, inputs, hidden, layers, dropout):
            super().__init__()
            self.steps, self.channels = 9, len(WEATHER)
            self.sequence_size = self.steps * self.channels
            self.gru = torch.nn.GRU(self.channels, hidden, num_layers=layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
            self.head = torch.nn.Sequential(torch.nn.Linear(hidden + inputs - self.sequence_size, hidden), torch.nn.ReLU(), torch.nn.Dropout(dropout), torch.nn.Linear(hidden, 1))

        def forward(self, x):
            sequence = x[:, :self.sequence_size].reshape(-1, self.steps, self.channels)
            _, state = self.gru(sequence)
            return self.head(torch.cat([state[-1], x[:, self.sequence_size:]], dim=1)).squeeze(-1)


def net_array(X):
    return X[[*SEQ_COLUMNS, *OTHER_COLUMNS]].to_numpy(np.float32)


def train_net(kind, X_train, y_train, epochs, settings, X_stop=None, y_stop=None):
    torch.manual_seed(SEED)
    scaler = StandardScaler().fit(X_train)
    data = torch.tensor(scaler.transform(X_train), dtype=torch.float32)
    labels = torch.tensor(y_train, dtype=torch.float32)
    builder = PoissonGRU if kind == 'gru' else PoissonMLP
    model = builder(data.shape[1], settings['hidden'], settings['layers'], settings['dropout'])
    optimizer = torch.optim.AdamW(model.parameters(), lr=settings['lr'], weight_decay=settings['weight_decay'])
    loss_function = torch.nn.PoissonNLLLoss(log_input=True, full=False)
    generator = torch.Generator().manual_seed(SEED)
    stop_data = None if X_stop is None else torch.tensor(scaler.transform(X_stop), dtype=torch.float32)
    stop_labels = None if y_stop is None else torch.tensor(y_stop, dtype=torch.float32)
    best_loss, best_epoch, waited = np.inf, 1, 0
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(len(data), generator=generator)
        for start in range(0, len(order), settings['batch']):
            batch = order[start:start + settings['batch']]
            optimizer.zero_grad()
            loss = loss_function(model(data[batch]), labels[batch])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
        if stop_data is not None:
            model.eval()
            with torch.no_grad():
                current = float(loss_function(model(stop_data), stop_labels))
            if current < best_loss - 1e-5:
                best_loss, best_epoch, waited = current, epoch, 0
            else:
                waited += 1
                if waited >= PATIENCE:
                    break
    return model, scaler, best_epoch


def fit_net(kind, Xf, yf, Xe, ye, Xv, params):
    if torch is None:
        raise RuntimeError('torch is not available')
    settings = {'hidden': 64, 'layers': 2, 'dropout': 0.1, 'lr': 1e-3, 'weight_decay': 1e-4, 'batch': 512, **params}
    A, B, C = net_array(Xf), net_array(Xe), net_array(Xv)
    _, _, best_epoch = train_net(kind, A, yf, MAX_EPOCHS, settings, B, ye)
    model, scaler, _ = train_net(kind, np.vstack([A, B]), np.concatenate([yf, ye]), best_epoch, settings)
    model.eval()
    with torch.no_grad():
        log_rate = model(torch.tensor(scaler.transform(C), dtype=torch.float32)).numpy()
    return np.exp(np.clip(log_rate, -20, 8))


def fit_mlp(Xf, yf, Xe, ye, Xv, params):
    return fit_net('mlp', Xf, yf, Xe, ye, Xv, params)


def fit_gru(Xf, yf, Xe, ye, Xv, params):
    return fit_net('gru', Xf, yf, Xe, ye, Xv, params)


CANDIDATES = {
    'Poisson GLM': {'family': 'Count GLM', 'rank': 1, 'fit': fit_poisson_glm},
    'Tweedie GLM': {'family': 'Count GLM', 'rank': 2, 'fit': fit_tweedie_glm},
    'Negative Binomial GLM': {'family': 'Count GLM', 'rank': 3, 'fit': fit_negative_binomial},
    'Poisson GLM + splines': {'family': 'GLM + splines', 'rank': 4, 'fit': fit_spline_glm},
    'Random Forest': {'family': 'Bagged trees', 'rank': 5, 'fit': fit_random_forest},
    'Extra Trees': {'family': 'Bagged trees', 'rank': 5, 'fit': fit_extra_trees},
    'HistGradientBoosting': {'family': 'Boosting', 'rank': 6, 'fit': fit_hist_gb},
    'LightGBM Poisson': {'family': 'Boosting', 'rank': 7, 'fit': fit_lgb_poisson},
    'LightGBM Tweedie': {'family': 'Boosting', 'rank': 7, 'fit': fit_lgb_tweedie},
    'XGBoost Poisson': {'family': 'Boosting', 'rank': 8, 'fit': fit_xgboost},
    'CatBoost Poisson': {'family': 'Boosting', 'rank': 8, 'fit': fit_catboost},
    'LightGBM Hurdle': {'family': 'Zero-aware', 'rank': 9, 'fit': fit_lgb_hurdle},
    'Poisson MLP': {'family': 'Neural', 'rank': 10, 'fit': fit_mlp},
    'Poisson GRU': {'family': 'Neural', 'rank': 11, 'fit': fit_gru},
}
print(len(CANDIDATES), 'candidate models registered')

## Segment 7: Validation folds, metrics and selection rule
Each fold trains on earlier years only. The last training year stops boosting and neural models, then they are refit on all training years.

In [ ]:
def split(horizon, year):
    frame = FRAMES[horizon]
    target = frame['target_date']
    fit_rows = target <= pd.Timestamp(year - 2, 12, 31)
    stop_rows = (target >= pd.Timestamp(year - 1, 1, 1) + PURGE) & (target <= pd.Timestamp(year - 1, 12, 31))
    val_rows = (target >= pd.Timestamp(year, 1, 1) + PURGE) & (target <= pd.Timestamp(year, 12, 31))
    fit_part, stop_part, val_part = frame[fit_rows], frame[stop_rows], frame[val_rows]
    assert fit_part['target_date'].max() < stop_part[DATE].min()
    assert stop_part['target_date'].max() < val_part[DATE].min()
    return fit_part, stop_part, val_part


def score(actual, predicted, cuts):
    actual = np.asarray(actual, dtype=float)
    predicted = np.nan_to_num(np.asarray(predicted, dtype=float), nan=DEV_FLOOR, posinf=1e4)
    predicted = np.clip(predicted, 0, None)
    true_high, predicted_high = actual >= cuts[1], predicted >= cuts[1]
    hits = int((true_high & predicted_high).sum())
    recall = hits / max(int(true_high.sum()), 1)
    precision = hits / max(int(predicted_high.sum()), 1)
    return {
        'deviance': float(mean_poisson_deviance(actual, np.clip(predicted, DEV_FLOOR, None))),
        'mae': float(mean_absolute_error(actual, predicted)),
        'rmse': float(np.sqrt(mean_squared_error(actual, predicted))),
        'hc_recall': recall,
        'hc_precision': precision,
        'hc_f1': 2 * recall * precision / max(recall + precision, 1e-9),
    }


def fold_cuts(fit_part, stop_part):
    labels = np.concatenate([fit_part['target'].to_numpy(float), stop_part['target'].to_numpy(float)])
    return [float(value) for value in np.quantile(labels, TIER_QUANTILES)]


def baseline_predictions(train, val):
    weeks = lambda f: f['target_date'].dt.isocalendar().week.astype(int)
    ward_mean = train.groupby(WARD)['target'].mean()
    ward_pred = val[WARD].map(ward_mean).fillna(float(train['target'].mean())).to_numpy(float)
    ward_week = train.assign(week_number=weeks(train)).groupby([WARD, 'week_number'])['target'].mean()
    keys = pd.MultiIndex.from_arrays([val[WARD], weeks(val)], names=[WARD, 'week_number'])
    seasonal = ward_week.reindex(keys).to_numpy(float)
    seasonal = np.where(np.isnan(seasonal), ward_pred, seasonal)
    return {
        'Baseline: ward mean': ward_pred,
        'Baseline: ward x week mean': seasonal,
        'Baseline: persistence': val['cases_t'].to_numpy(float),
        'Baseline: same week last year': val['last_year'].to_numpy(float),
    }


def slug(text):
    return re.sub('[^a-z0-9]+', '_', text.lower()).strip('_')


def evaluate(name, fit, params, stage, years=None):
    path = CV_DIR / f'{stage}__{slug(name)}.json'
    if path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    rows = []
    for horizon in HORIZONS:
        for year in (years or VAL_YEARS):
            fit_part, stop_part, val_part = split(horizon, year)
            start = time.perf_counter()
            predicted = fit(fit_part[FEATURES], fit_part['target'].to_numpy(float), stop_part[FEATURES], stop_part['target'].to_numpy(float), val_part[FEATURES], params)
            seconds = time.perf_counter() - start
            rows.append({'model': name, 'horizon': horizon, 'fold': year, 'seconds': seconds, **score(val_part['target'], predicted, fold_cuts(fit_part, stop_part))})
    path.write_text(json.dumps(rows), encoding='utf-8')
    return rows


def evaluate_baselines(years=None):
    rows = []
    for horizon in HORIZONS:
        for year in (years or VAL_YEARS):
            fit_part, stop_part, val_part = split(horizon, year)
            train = pd.concat([fit_part, stop_part], ignore_index=True)
            cuts = fold_cuts(fit_part, stop_part)
            for name, predicted in baseline_predictions(train, val_part).items():
                rows.append({'model': name, 'horizon': horizon, 'fold': year, 'seconds': 0.0, **score(val_part['target'], predicted, cuts)})
    return rows


def summarize(rows, baseline_rows):
    data = pd.DataFrame(rows)
    best = pd.DataFrame(baseline_rows).groupby(['horizon', 'fold'])['mae'].min().rename('best_baseline_mae').reset_index()
    data = data.merge(best, on=['horizon', 'fold'])
    data['beats_baseline'] = data['mae'] < data['best_baseline_mae']
    records = []
    for name, group in data.groupby('model', sort=False):
        by_horizon = group.groupby('horizon')[['mae', 'best_baseline_mae']].mean()
        passes = bool((by_horizon['mae'] < by_horizon['best_baseline_mae']).all() and group['beats_baseline'].mean() >= 0.75)
        fold_deviance = group.groupby('fold')['deviance'].mean()
        records.append({
            'model': name, 'deviance': float(group['deviance'].mean()), 'worst_fold_deviance': float(fold_deviance.max()),
            'mae': float(group['mae'].mean()), 'rmse': float(group['rmse'].mean()),
            'hc_recall': float(group['hc_recall'].mean()), 'hc_precision': float(group['hc_precision'].mean()), 'hc_f1': float(group['hc_f1'].mean()),
            'cells_beating_baseline': float(group['beats_baseline'].mean()), 'passes_baselines': passes,
            'seconds_per_fit': float(group['seconds'].mean()),
        })
    return pd.DataFrame(records).sort_values('deviance').reset_index(drop=True)


def choose(table, rows, ranks):
    eligible = table[table['passes_baselines']]
    if eligible.empty:
        return None, pd.DataFrame()
    by_fold = pd.DataFrame(rows).groupby(['model', 'fold'])['deviance'].mean().unstack('fold')
    leader = eligible.iloc[0]['model']
    records = []
    for name in eligible['model']:
        gap = by_fold.loc[name] - by_fold.loc[leader]
        standard_error = float(gap.std(ddof=1) / np.sqrt(len(gap))) if len(gap) > 1 else 0.0
        mean_gap = float(gap.mean())
        records.append({'model': name, 'gap_to_leader': mean_gap, 'standard_error': standard_error, 'tied_with_leader': mean_gap <= standard_error, 'simplicity_rank': ranks[name]})
    gaps = pd.DataFrame(records)
    winner = gaps[gaps['tied_with_leader']].sort_values(['simplicity_rank', 'gap_to_leader']).iloc[0]['model']
    return winner, gaps


for horizon in HORIZONS:
    for year in VAL_YEARS:
        fit_part, stop_part, val_part = split(horizon, year)
        print(f'h={horizon} fold {year}: fit {len(fit_part):,} | stop {len(stop_part):,} | validate {len(val_part):,}')

## Segment 8: Baselines

In [ ]:
BASELINE_ROWS = evaluate_baselines()
baseline_table = pd.DataFrame(BASELINE_ROWS).groupby('model')[['deviance', 'mae', 'rmse', 'hc_recall', 'hc_precision', 'hc_f1']].mean().sort_values('mae')
display(baseline_table.round(3))

## Segment 9: Compare all models with default settings
A model that fails is recorded and skipped, so the run continues.

In [ ]:
SCREEN_ROWS, FAILED = [], {}
for name, spec in CANDIDATES.items():
    start = time.perf_counter()
    try:
        rows = evaluate(name, spec['fit'], {}, 'screen')
        SCREEN_ROWS += rows
        mean_deviance = np.mean([row['deviance'] for row in rows])
        print(f'{name:<26} deviance {mean_deviance:.4f} | {time.perf_counter() - start:.0f}s')
    except Exception as exc:
        FAILED[name] = f'{type(exc).__name__}: {str(exc)[:200]}'
        print(f'{name:<26} FAILED: {FAILED[name]}')

if not SCREEN_ROWS:
    raise RuntimeError(f'Every model failed: {FAILED}')
RANKS = {name: spec['rank'] for name, spec in CANDIDATES.items()}
screen_table = summarize(SCREEN_ROWS, BASELINE_ROWS)
display(screen_table.round(3))

plot_table = pd.concat([screen_table[['model', 'mae']], baseline_table.reset_index()[['model', 'mae']]]).sort_values('mae', ascending=False)
colors = ['#e0683a' if label.startswith('Baseline') else '#2a78d6' for label in plot_table['model']]
plt.figure(figsize=(9, 0.32 * len(plot_table) + 1))
plt.barh(plot_table['model'], plot_table['mae'], color=colors)
plt.xlabel('Mean absolute error across folds and horizons (blue = model, orange = baseline)')
plt.tight_layout()
plt.show()

## Segment 10: Pick the top 3 finalists
The three lowest-deviance models that beat the baselines move on to tuning.

In [ ]:
eligible_models = screen_table.loc[screen_table['passes_baselines'], 'model'].tolist()
FINALISTS = eligible_models[:3]
print('Models that beat the baselines:', len(eligible_models), 'of', len(screen_table))
print('Finalists:', FINALISTS if FINALISTS else 'NONE')
if FAILED:
    print('Models that failed to run:', list(FAILED))
if not FINALISTS:
    print('No model beat the baselines. Send this output back before continuing.')

## Segment 11: Hyperparameter tuning with Optuna
Each finalist gets the same trial and time budget. Weak trials are stopped early. One parameter set is tuned across all horizons to reduce overfitting.

In [ ]:
def lgb_space(trial):
    return {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 7, 63),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 200, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 30.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
    }


def lgb_tweedie_space(trial):
    return {**lgb_space(trial), 'tweedie_variance_power': trial.suggest_float('tweedie_variance_power', 1.05, 1.9)}


def xgb_space(trial):
    return {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 50.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 30.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
    }


def catboost_space(trial):
    return {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 9),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.5, 30.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'rsm': trial.suggest_float('rsm', 0.5, 1.0),
    }


def hgb_space(trial):
    return {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 7, 63),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 10, 200, log=True),
        'l2_regularization': trial.suggest_float('l2_regularization', 1e-3, 30.0, log=True),
    }


def forest_space(trial):
    return {
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 50, log=True),
        'max_features': trial.suggest_float('max_features', 0.2, 1.0),
        'max_samples': trial.suggest_float('max_samples', 0.4, 1.0),
    }


def glm_space(trial):
    return {'alpha': trial.suggest_float('alpha', 1e-5, 1.0, log=True)}


def tweedie_glm_space(trial):
    return {**glm_space(trial), 'power': trial.suggest_float('power', 1.05, 1.9)}


def spline_space(trial):
    return {**glm_space(trial), 'n_knots': trial.suggest_int('n_knots', 3, 8)}


def net_space(trial):
    return {
        'hidden': trial.suggest_categorical('hidden', [32, 64, 128]),
        'layers': trial.suggest_int('layers', 1, 3),
        'dropout': trial.suggest_float('dropout', 0.0, 0.4),
        'lr': trial.suggest_float('lr', 1e-4, 5e-3, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True),
    }


SPACES = {
    'Poisson GLM': glm_space, 'Tweedie GLM': tweedie_glm_space, 'Negative Binomial GLM': glm_space,
    'Poisson GLM + splines': spline_space, 'Random Forest': forest_space, 'Extra Trees': forest_space,
    'HistGradientBoosting': hgb_space, 'LightGBM Poisson': lgb_space, 'LightGBM Tweedie': lgb_tweedie_space,
    'XGBoost Poisson': xgb_space, 'CatBoost Poisson': catboost_space, 'LightGBM Hurdle': lgb_space,
    'Poisson MLP': net_space, 'Poisson GRU': net_space,
}


def tune(name):
    result_path = TUNE_DIR / f'{slug(name)}.json'
    if result_path.exists():
        return json.loads(result_path.read_text(encoding='utf-8'))
    fit, space = CANDIDATES[name]['fit'], SPACES[name]
    saved_db = TUNE_DIR / f'{slug(name)}.db'
    local_db = Path.cwd() / f'{RUN_DIR.name}_{slug(name)}_study.db'
    if saved_db.exists():
        shutil.copy(saved_db, local_db)
    study = optuna.create_study(
        study_name=slug(name), storage=f'sqlite:///{local_db.as_posix()}', load_if_exists=True, direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=1),
    )

    def objective(trial):
        params, fold_scores = space(trial), []
        for step, year in enumerate(VAL_YEARS):
            values = []
            for horizon in HORIZONS:
                fit_part, stop_part, val_part = split(horizon, year)
                predicted = fit(fit_part[FEATURES], fit_part['target'].to_numpy(float), stop_part[FEATURES], stop_part['target'].to_numpy(float), val_part[FEATURES], params)
                values.append(score(val_part['target'], predicted, fold_cuts(fit_part, stop_part))['deviance'])
            fold_scores.append(float(np.mean(values)))
            trial.report(float(np.mean(fold_scores)), step)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(fold_scores))

    def checkpoint(_study, _trial):
        shutil.copy(local_db, saved_db)

    finished = [t for t in study.trials if t.state.is_finished()]
    remaining = max(N_TRIALS - len(finished), 0)
    if remaining:
        study.optimize(objective, n_trials=remaining, timeout=TUNE_MINUTES * 60, callbacks=[checkpoint], catch=(Exception,), show_progress_bar=False)
    states = [t.state.name for t in study.trials]
    completed = states.count('COMPLETE')
    if completed == 0:
        raise RuntimeError(f'No tuning trial completed for {name}')
    result = {'model': name, 'best_params': study.best_params, 'best_cv_deviance': study.best_value, 'completed': completed, 'pruned': states.count('PRUNED'), 'failed': states.count('FAIL')}
    out_of_budget = len([t for t in study.trials if t.state.is_finished()]) < N_TRIALS
    result['stopped_by_time_budget'] = out_of_budget
    result_path.write_text(json.dumps(result), encoding='utf-8')
    return result


TUNING = {}
if optuna is None:
    print('Optuna is unavailable, so tuning is skipped.')
else:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    for name in FINALISTS:
        start = time.perf_counter()
        try:
            TUNING[name] = tune(name)
            item = TUNING[name]
            print(f"{name}: best deviance {item['best_cv_deviance']:.4f} | {item['completed']} complete, {item['pruned']} pruned | {time.perf_counter() - start:.0f}s")
            print('  ', item['best_params'])
        except Exception as exc:
            FAILED[f'{name} tuning'] = f'{type(exc).__name__}: {str(exc)[:200]}'
            print(f"{name} tuning FAILED: {FAILED[f'{name} tuning']}")

## Segment 12: Final model decision
Default and tuned finalists are compared with the rule fixed at the top. If tuning gives no meaningful gain, the simpler default version wins.

In [ ]:
FINAL_ROWS = [row for row in SCREEN_ROWS if row['model'] in FINALISTS]
FINAL_SPECS = {name: {'base': name, 'params': {}} for name in FINALISTS}
FINAL_RANKS = {name: RANKS[name] for name in FINALISTS}

for name, item in TUNING.items():
    tuned_name = f'{name} (tuned)'
    try:
        FINAL_ROWS += evaluate(tuned_name, CANDIDATES[name]['fit'], item['best_params'], 'tuned')
        FINAL_SPECS[tuned_name] = {'base': name, 'params': item['best_params']}
        FINAL_RANKS[tuned_name] = RANKS[name] + 0.5
    except Exception as exc:
        FAILED[tuned_name] = f'{type(exc).__name__}: {str(exc)[:200]}'

WINNER, final_gaps, final_table = None, pd.DataFrame(), pd.DataFrame()
if FINAL_ROWS:
    final_table = summarize(FINAL_ROWS, BASELINE_ROWS)
    WINNER, final_gaps = choose(final_table, FINAL_ROWS, FINAL_RANKS)
    display(final_table.round(4))
    display(final_gaps.round(4))

print('SELECTED MODEL:', WINNER if WINNER else 'NONE')
if WINNER:
    print('Base model:', FINAL_SPECS[WINNER]['base'])
    print('Parameters:', FINAL_SPECS[WINNER]['params'] or 'default settings')
    winner_rows = pd.DataFrame([row for row in FINAL_ROWS if row['model'] == WINNER])
    print('Per horizon:')
    display(winner_rows.groupby('horizon')[['deviance', 'mae', 'rmse', 'hc_recall', 'hc_precision']].mean().round(3))
    print('Per fold year:')
    display(winner_rows.groupby('fold')[['deviance', 'mae', 'rmse', 'hc_recall', 'hc_precision']].mean().round(3))

## Segment 13: One-time confirmation on 2023
Only the selected model and baselines are scored. This checks the decision and **must not be used to switch models**.

In [ ]:
CONFIRMATION = None
if WINNER:
    spec = FINAL_SPECS[WINNER]
    model_rows = evaluate(WINNER, CANDIDATES[spec['base']]['fit'], spec['params'], 'confirm_2023', years=(2023,))
    base_rows = evaluate_baselines(years=(2023,))
    confirm = pd.DataFrame([*model_rows, *base_rows])
    display(confirm.pivot_table(index='model', columns='horizon', values='mae').round(3))
    best_base = pd.DataFrame(base_rows).groupby('horizon')['mae'].min()
    model_mae = pd.DataFrame(model_rows).set_index('horizon')['mae']
    improvement = (1 - model_mae / best_base) * 100
    passed = bool((model_mae < best_base).all())
    CONFIRMATION = {
        'passed': passed,
        'model_rows': model_rows,
        'best_baseline_mae': {str(k): float(v) for k, v in best_base.items()},
        'improvement_pct': {str(k): float(v) for k, v in improvement.items()},
    }
    print('Improvement over the best baseline (%):', improvement.round(1).to_dict())
    print('CONFIRMED' if passed else 'NOT CONFIRMED: do not ship without review')

## Segment 14: Save the report
Send back `model_selection_report.json`. It contains results only, with no raw data.

In [ ]:
def versions():
    import importlib.metadata as metadata

    found = {'python': sys.version.split()[0]}
    for package in ['numpy', 'pandas', 'scikit-learn', 'lightgbm', 'xgboost', 'catboost', 'statsmodels', 'optuna', 'torch']:
        try:
            found[package] = metadata.version(package)
        except metadata.PackageNotFoundError:
            found[package] = None
    return found


report = {
    'valid_result': not QUICK_RUN,
    'mode': 'quick_smoke_test' if QUICK_RUN else 'full',
    'versions': versions(),
    'settings': {'horizons': list(HORIZONS), 'folds': list(VAL_YEARS), 'trials': N_TRIALS, 'minutes_per_finalist': TUNE_MINUTES, 'max_trees': MAX_TREES},
    'final_test_rows_removed': hidden_rows,
    'diagnostics': DIAGNOSTICS,
    'baselines': baseline_table.reset_index().to_dict('records'),
    'screening': screen_table.to_dict('records'),
    'screening_detail': SCREEN_ROWS,
    'failed': FAILED,
    'finalists': FINALISTS,
    'tuning': TUNING,
    'final_comparison': final_table.to_dict('records') if not final_table.empty else [],
    'tie_analysis': final_gaps.to_dict('records') if not final_gaps.empty else [],
    'winner': WINNER,
    'winner_spec': FINAL_SPECS.get(WINNER) if WINNER else None,
    'confirmation_2023': CONFIRMATION,
}


def clean(value):
    if isinstance(value, dict):
        return {str(k): clean(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean(v) for v in value]
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        return None if not np.isfinite(value) else round(float(value), 5)
    return value


report_path = RUN_DIR / 'model_selection_report.json'
report_path.write_text(json.dumps(clean(report), indent=1), encoding='utf-8')
print('Report saved:', report_path)
print('VALID RESULT' if not QUICK_RUN else 'SMOKE TEST ONLY: set QUICK_RUN = False and Run all')
print('Winner:', WINNER)
print('Failed:', list(FAILED) if FAILED else 'none')
if IN_COLAB:
    from google.colab import files

    files.download(str(report_path))